# Belajar alur AES: teks → empat skor

Default **inspect**. Data draf dapat diperiksa tanpa skor guru. Angka dari head baru diberi label **UNTRAINED**, bukan penilaian siswa. Output selalu `[isi, organisasi, kebahasaan, mekanik]`, skala 1–5.

Pada Colab, letakkan seluruh proyek di satu folder (melalui Drive atau upload arsip proyek), lalu ubah ROOT di cell berikut. Path dapat diganti; notebook tidak membutuhkan file sesi lama. Untuk Python/R lengkap jalankan `bash scripts/setup_colab.sh` dari root proyek. Notebook memasang dependency Python hanya bila modul yang diperlukan belum tersedia.

## Status sumber proposal dan paper

Proposal terbaru telah dibaca dan salinan/hash disimpan di `project_sources/` serta `outputs/source_review/review.json`. File terbaru sudah memakai empat trait; beberapa pengaturan (window BGE, head MLP S1/S3, 10 epoch, tiga seed/final ensemble) berbeda dari kontrak aktif. Notebook ini tetap mengikuti instruksi pengguna: BGE truncation 1024, S1/S3 linear, maksimum 5 epoch, 25 run, satu final seed42. Learning rate dan regularisasi kini terverifikasi di Tabel 3.4.

Lihat [pencocokan sumber](../docs/source_reconciliation.md). Paper sudah tersedia dari ZIP. Seluruh 4.051 halaman diindeks; bagian metode/evaluasi relevan dibaca dan dicocokkan di `docs/paper_review.md`. Dense CLS/L2, batas QWK, dan dependensi fold mendukung catatan implementasi; tidak mengisi label atau membuktikan keunggulan S3. Cakupan pembacaan bukan setiap paragraf seluruh bundel. Kontrak empat trait dan status label/rubrik tetap berlaku. Path dokumen dapat diatur melalui `sources` pada config.


In [ ]:
from pathlib import Path
import os, sys, importlib.util, subprocess
# Ubah ROOT jika lokasi proyek berbeda. Otomatis mencari dari cwd dan parent.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p/'configs/default.json').exists()), None)
if ROOT is None:
    ROOT = Path('/content/Automated_Essay_Scoring')  # konfigurasi Colab yang dapat diganti
if not (ROOT/'configs/default.json').exists():
    raise FileNotFoundError(f'Folder proyek belum ditemukan: {ROOT}. Atur ROOT ke folder proyek lengkap.')
os.chdir(ROOT)
os.environ.setdefault('HF_HOME', str(ROOT/'outputs/cache/huggingface'))
os.environ.setdefault('OMP_NUM_THREADS', '2')
os.environ.setdefault('MKL_NUM_THREADS', '2')
required = ['torch','transformers','FlagEmbedding','pandas','scipy','sklearn']
if any(importlib.util.find_spec(m) is None for m in required):
    subprocess.run([sys.executable,'-m','pip','install','-r','requirements.txt'],check=True)
sys.path.insert(0, str(ROOT/'src'))
MODE = 'inspect'  # inspect | learning | proposal
RUN_ENCODERS = True  # checkpoint asli; unduhan BGE ~2.3GB, RoBERTa ~0.5GB
from aes.io import load_config
cfg = load_config(ROOT/'configs/default.json')
# Alternatif: cfg['csv'] = '/content/dataset_aes_multitrait_DRAF.csv'
# Alternatif: cfg['rubric'] = '/content/rubrik_multitrait_DRAF.json'
missing_inputs = [cfg[k] for k in ['csv','rubric'] if not Path(cfg[k]).is_file()]
if missing_inputs:
    raise FileNotFoundError('Input privat belum tersedia: '+', '.join(missing_inputs)+'. Atur cfg[\'csv\']/cfg[\'rubric\']; lihat README. Repo publik tidak memuat data siswa.')
print('Root:', ROOT, 'Mode:', MODE)


## 1. Audit aktual dan kelayakan
Loader membaca CSV dengan UTF-8 BOM dan mempertahankan semua kolom. Skor kosong tetap missing. Audit memeriksa versi rubrik, status, label, duplikasi, dan jumlah kelompok. Group usulan tidak membuktikan identitas. Kebutuhan supervised ditampilkan tanpa menghentikan inspeksi.

In [ ]:
import json
from aes.inspection import inspect_data, inspect_models
from aes.audit import validate_inputs_for_training, EligibilityError
from aes.io import sha256
import pandas as pd
pd.set_option('display.max_colwidth', 100)
df, rubric, report = inspect_data(cfg)
print(json.dumps(report, ensure_ascii=False, indent=2))
print('Supervised belum layak:' if report['training_blockers'] else 'Syarat data terpenuhi:')
for reason in report['training_blockers']: print('-', reason)


## 2. Satu esai dan rubrik asli
Deskriptor diambil langsung dari JSON. Nilai huruf/provenance tidak masuk fitur. Best fit dan memilih deskriptor lebih rendah ketika sama kuat adalah aturan guru, bukan algoritme mengisi label.

In [ ]:
essay = df.iloc[0]
print('essay_id:', essay.essay_id)
print(essay[['scoring_status','transcription_status','text_completeness','identity_status','group_status','ready_for_training']].to_dict())
print(essay.text_raw)
for trait in rubric['traits']:
    print('\nTRAIT:', trait['name'], trait['categories'])
    print('Definisi:', trait['definition'])
    print('Indikator:', trait['indicators'])
    print('Dikecualikan:', trait['exclude'])
    for level, descriptor in trait['levels'].items(): print(level, descriptor)
print('Aturan guru:', rubric['scoring_rule'])
print('Missing:', rubric['missing_rule'])


## 3. Cleaning yang bisa diperiksa
NFC menyamakan representasi Unicode; newline diseragamkan dan artefak kontrol dihapus. Paragraf, kapital, salah ejaan, imbuhan, tanda baca tetap. Tidak menggabungkan baris tanpa bukti. Normalisasi kutip/dash/elipsis default mati karena bisa memengaruhi mekanik.

In [ ]:
from aes.cleaning import clean
from difflib import unified_diff
text_model = clean(essay.text_raw, **cfg['preprocessing'])
print('Sebelum:', repr(essay.text_raw[:300]))
print('Sesudah:', repr(text_model[:300]))
print('Berubah:', text_model != essay.text_raw)
print(''.join(unified_diff(essay.text_raw.splitlines(True), text_model.splitlines(True), fromfile='raw', tofile='model')))
print('Log cleaning:', Path(cfg['output'])/'cleaning_review.csv')


## 4. Tokenisasi dan 5. embedding
Token IDs adalah nomor kamus; embedding adalah representasi konteks. BGE memotong di 1024 termasuk special tokens; tidak memakai window. RoBERTa mencakup seluruh teks melalui window 512 dengan overlap 128 token isi. Capacity dihitung dari tokenizer, bukan stride hard-code. Checkpoint ini memakai tokenizer RoBERTa slow karena fast artifact tidak konsisten menghitung special tokens.

Cell berikut menjalankan encoder asli dan hanya mencetak beberapa elemen vektor. Error jaringan dilaporkan sebagai blocker; error kode/shape/dependency lain tetap muncul. Tidak mengisi CSV atau menghitung loss dari skor kosong.

In [ ]:
from requests.exceptions import RequestException
from huggingface_hub.errors import HfHubHTTPError
model_result = None
if RUN_ENCODERS:
    try:
        model_result = inspect_models(cfg, df)
    except (RequestException, HfHubHTTPError) as exc:
        print('CHECKPOINT BLOCKED:', type(exc).__name__, str(exc)[:500])
        print('Inspeksi data tetap tersedia. Periksa akses Hugging Face; tidak memakai checkpoint pengganti.')
else:
    print('Encoder belum dijalankan: RUN_ENCODERS=False.')
if model_result is not None:
    print('Revision/config:', model_result['resolved_models'])
    print('BGE tokenisasi tiap esai:', model_result['bge_tokenization'])
    print('RoBERTa window tiap esai:', model_result['roberta_windows'])
    print('Jumlah esai terpotong:', model_result['truncated_essays'], 'token dipotong:', model_result['removed_tokens'])
    print('Contoh token IDs terpisah:', model_result['token_ids_by_encoder'])
    print('Frozen/trainable aktual:', model_result['gradient_policy'])
    print('Contoh elemen vektor:', model_result['vector_first_five'])


## 6–8. Shapes dan forward UNTRAINED
Untuk satu esai: BGE `[1,1024]`; mean CLS RoBERTa `[1,768]`; projection `[1,256]`; concat `[1,1024]`; output `[1,4]`. Head baru belum belajar skor guru. Angka diagnostik bisa di luar 1–5 dan tidak boleh digunakan sebagai penilaian.

In [ ]:
if model_result is not None:
    print('SHAPES:', model_result['shapes'])
    print('UNTRAINED — BUKAN PENILAIAN ESAI:', model_result['untrained_output'])
else:
    print('Shapes aktual belum tersedia karena encoder belum berhasil dijalankan.')


## 9. Frozen, trainable, dan aliran gradien
BGE frozen: eval, requires_grad=False, encode tanpa graph; hanya dense-nya boleh dicache. RoBERTa fine-tuned: hidden state dan mean window tetap dalam graph; encoder masuk optimizer. S3 projection, LayerNorm, dan head ikut belajar. Empat trait berbagi encoder/loss sehingga error tidak otomatis independen.

Tes di bawah memakai **fixture teknis buatan**, bukan dataset alternatif, bukan label CSV, bukan metrik penelitian. Tes memeriksa shapes B=1/B=2, gradien, reload, window dan metrik. Tidak membutuhkan unduhan model.

In [ ]:
completed = subprocess.run([sys.executable,'-m','pytest','-q'],check=True)
print('Unit checks selesai; ini bukan training penelitian.')


## 10. Langkah supervised setelah guru menilai
Pastikan transkripsi lengkap/verified, empat skor guru 1–5, rubrik disahkan guru/tanggal, identitas dan hubungan duplikasi diperiksa. Isi review terhadap hash CSV yang tepat. Jangan menaikkan flag hanya agar kode lolos. Mode learning satu epoch train satu fold; mode proposal tuning + 25 CV + refit + final test. Default inspect tidak menjalankan keduanya.

In [ ]:
review = json.loads(Path(cfg['split']['review_file']).read_text())
try:
    labels = validate_inputs_for_training(df, rubric, review, sha256(cfg['csv']))
except EligibilityError as exc:
    labels = None
    print('SUPERVISED SKIP — data belum layak:\n', exc)
if MODE in ['learning','proposal'] and labels is not None:
    from aes.experiment import run
    result = run(cfg, mode=MODE)
    print('Laporan disimpan terpisah sesuai mode:', MODE)
elif MODE == 'inspect':
    print('Mode inspect selesai. Tidak ada loss/evaluasi penelitian dari label kosong.')
elif MODE not in ['learning','proposal']:
    raise ValueError('MODE tidak dikenal')


## Cara membaca hasil
RMSE/Pearson memakai prediksi kontinu mentah; QWK/AC2 memakai salinan kategori setelah clipping dan nearest category, midpoint ke bawah. Tidak flatten trait. Undefined tidak menjadi nol. AC2 perlu pemeriksaan referensi R: `python scripts/verify_ac2.py --require-r`; setup R/irrCAC ada di `scripts/setup_colab.sh`. Sesudah checkpoint sungguh dilatih, gunakan `python -m aes.cli predict --checkpoint ... --file esai.txt`; inference tidak membutuhkan label guru.

Lihat `docs/implementation_contract.md` untuk asumsi, perbedaan proposal, split group, monitor/fallback, dan batas bukti.